# Notebook 03 — Complete 25-Quiz Practice Lab (CTEs, Subqueries, Temp Tables & Procedures)

**Session:** Database - SQL (3)  
**Module:** 02 · Calculus and Linear Algebra

## Goal
Execute and verify all **25 practice quizzes** across the `sql_hr`, `sql_invoicing`, and `sql_store` domains using `../data/sql_store_seed.sql`, with automated assertions validating every result.

In [ ]:
import sqlite3
from pathlib import Path

SEED_PATH = Path("../data/sql_store_seed.sql")
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")
conn.executescript(SEED_PATH.read_text(encoding="utf-8"))


def query(sql: str, params: tuple = (), connection: sqlite3.Connection = conn) -> list[dict]:
    """Execute a SQL statement and return rows as a list of dictionaries."""
    cur = connection.execute(sql, params)
    cols = [desc[0] for desc in cur.description]
    return [dict(zip(cols, row, strict=True)) for row in cur.fetchall()]


def show_rows(rows: list[dict], limit: int | None = None) -> None:
    """Pretty-print query results as an aligned ASCII table."""
    display_rows = rows if limit is None else rows[:limit]
    if not display_rows:
        print("(0 rows)")
        return
    cols = list(display_rows[0].keys())
    widths = {c: max(len(c), *(len(str(r[c])) for r in display_rows)) for c in cols}
    header = " | ".join(f"{c:<{widths[c]}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in display_rows:
        print(" | ".join(f"{r[c]!s:<{widths[c]}}" for c in cols))


print("Loaded tables:")
show_rows(query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;"))

## Quizzes 1–10: Subqueries, Temp Tables, CTEs, and Invoicing Procedures

In [ ]:
# Quiz 1 (sql_hr): Employees earning more than their direct manager
q1_sql = """
SELECT e1.employee_id, e1.first_name, e1.last_name, e1.salary
FROM employees e1
WHERE e1.salary > (
    SELECT e2.salary
    FROM employees e2
    WHERE e1.reports_to = e2.employee_id
);
"""
assert len(query(q1_sql)) == 0  # In baseline org chart, no direct report out-earns their manager

# Verify Quiz 1 under a temporary salary bump
conn.execute("SAVEPOINT q1_test;")
conn.execute("UPDATE employees SET salary = 150000 WHERE employee_id = 56274;")
q1_bumped = query(q1_sql)
assert [r["employee_id"] for r in q1_bumped] == [56274]
conn.execute("ROLLBACK TO SAVEPOINT q1_test;")
conn.execute("RELEASE SAVEPOINT q1_test;")
print("Quiz 1: PASS")

# Quiz 2 (sql_hr): Temp table temp_employee_office
conn.executescript("""
DROP TABLE IF EXISTS temp_employee_office;
CREATE TEMPORARY TABLE temp_employee_office AS
SELECT e.first_name, e.last_name, o.address AS office_address
FROM employees e
JOIN offices o ON e.office_id = o.office_id;
""")
q2 = query("SELECT * FROM temp_employee_office ORDER BY first_name;")
assert len(q2) == 8
print("Quiz 2: PASS")


# Quiz 3 (sql_invoicing): Stored procedure update_payment_invoice
def sp_update_payment_invoice(
    payment_id: int,
    client_id: int,
    invoice_id: int,
    date_str: str,
    amount: float,
    payment_method: int,
) -> None:
    conn.execute(
        "INSERT INTO payments (payment_id, client_id, invoice_id, date, amount, payment_method) VALUES (?, ?, ?, ?, ?, ?);",
        (payment_id, client_id, invoice_id, date_str, amount, payment_method),
    )
    conn.execute(
        "UPDATE invoices SET payment_total = ROUND(payment_total + ?, 2) WHERE invoice_id = ?;",
        (amount, invoice_id),
    )


conn.execute("SAVEPOINT q3_test;")
sp_update_payment_invoice(9, 3, 3, "2026-11-26", 147.99, 1)
assert (
    query("SELECT payment_total FROM invoices WHERE invoice_id = 3;")[0]["payment_total"] == 147.99
)
conn.execute("ROLLBACK TO SAVEPOINT q3_test;")
conn.execute("RELEASE SAVEPOINT q3_test;")
print("Quiz 3: PASS")


# Quiz 4 (sql_invoicing): Stored procedure top_paying_client_last_year
def sp_top_paying_client_last_year(as_of_date: str = "2019-12-31") -> list[dict]:
    return query(
        """
        SELECT client_id, COUNT(payment_id) AS payment_count
        FROM payments
        WHERE date >= date(?, '-1 year')
        GROUP BY client_id
        ORDER BY payment_count DESC, client_id ASC
        LIMIT 1;
        """,
        (as_of_date,),
    )


q4 = sp_top_paying_client_last_year("2019-12-31")
assert q4[0]["payment_count"] == 2
print("Quiz 4: PASS")

# Quiz 5 (sql_store): Orders with total amount > average order amount
q5 = query("""
SELECT o.order_id
FROM orders o
WHERE (
    SELECT SUM(oi.quantity * oi.unit_price)
    FROM order_items oi
    WHERE oi.order_id = o.order_id
) > (
    SELECT AVG(order_amount)
    FROM (
        SELECT oi.order_id, SUM(oi.quantity * oi.unit_price) AS order_amount
        FROM order_items oi
        GROUP BY oi.order_id
    ) AS average_order
)
ORDER BY o.order_id;
""")
assert [r["order_id"] for r in q5] == [3, 4, 6, 7, 10]
print("Quiz 5: PASS")

# Quiz 6 (sql_store): Most frequent product in orders using a CTE
q6 = query("""
WITH product_frequency AS (
    SELECT product_id, COUNT(*) AS order_count
    FROM order_items
    GROUP BY product_id
)
SELECT product_id, order_count
FROM product_frequency
ORDER BY order_count DESC, product_id ASC
LIMIT 1;
""")
assert q6[0]["product_id"] == 3 and q6[0]["order_count"] == 5
print("Quiz 6: PASS")

# Quiz 7 (sql_store): Temp table temp_customer_totals
conn.executescript("""
DROP TABLE IF EXISTS temp_customer_totals;
CREATE TEMPORARY TABLE temp_customer_totals AS
SELECT
    o.customer_id,
    SUM(oi.quantity) AS total_quantity,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS total_spent
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY o.customer_id;
""")
q7 = query("SELECT * FROM temp_customer_totals ORDER BY customer_id;")
assert len(q7) == 7
print("Quiz 7: PASS")

# Quiz 8 (sql_hr): Lowest-paid employee in each office using a CTE
q8 = query("""
WITH min_salary_office AS (
    SELECT office_id, MIN(salary) AS min_salary
    FROM employees
    GROUP BY office_id
)
SELECT e.first_name, e.last_name, e.office_id, e.salary
FROM employees e
JOIN min_salary_office mso
    ON e.office_id = mso.office_id
   AND e.salary = mso.min_salary
ORDER BY e.office_id;
""")
assert len(q8) == 2
assert q8[0]["first_name"] == "North" and q8[0]["salary"] == 45000
assert q8[1]["first_name"] == "Elena" and q8[1]["salary"] == 58000
print("Quiz 8: PASS")


# Quiz 9 (sql_invoicing): Stored procedure calculate_unpaid_amount
def sp_calculate_unpaid_amount() -> list[dict]:
    return query("""
    SELECT
        c.client_id,
        c.name,
        ROUND(
            COALESCE((SELECT SUM(i.invoice_total) FROM invoices i WHERE i.client_id = c.client_id), 0)
            - COALESCE((SELECT SUM(p.amount) FROM payments p WHERE p.client_id = c.client_id), 0),
            2
        ) AS unpaid_amount
    FROM clients c
    ORDER BY c.client_id;
    """)


q9 = sp_calculate_unpaid_amount()
assert len(q9) == 6
assert {r["client_id"]: r["unpaid_amount"] for r in q9}[4] == 180.45
print("Quiz 9: PASS")

# Quiz 10 (sql_invoicing): Top 5 clients by unpaid invoices using a subquery
q10 = query("""
SELECT
    c.client_id,
    c.name,
    ROUND(unpaid_amounts.unpaid_amount, 2) AS unpaid_amount
FROM clients c
JOIN (
    SELECT
        client_id,
        SUM(invoice_total) - COALESCE(SUM(payment_total), 0) AS unpaid_amount
    FROM invoices
    GROUP BY client_id
) AS unpaid_amounts
    ON c.client_id = unpaid_amounts.client_id
ORDER BY unpaid_amounts.unpaid_amount DESC, c.client_id ASC
LIMIT 5;
""")
assert len(q10) == 5
assert [r["client_id"] for r in q10[:3]] == [4, 3, 6]
print("Quiz 10: PASS")

## Quizzes 11–25: Revenue, Promotions, Discounts, Cursors, Anti-Joins & Division

In [ ]:
# Quiz 11 (sql_store): Total revenue per product using a subquery
q11 = query("""
SELECT
    p.product_id,
    p.name,
    ROUND(SUM(order_item_totals.total_amount), 2) AS total_revenue
FROM products p
JOIN (
    SELECT product_id, quantity * unit_price AS total_amount
    FROM order_items
) AS order_item_totals
    ON p.product_id = order_item_totals.product_id
GROUP BY p.product_id, p.name
ORDER BY total_revenue DESC;
""")
assert len(q11) == 11
assert q11[0]["product_id"] == 3  # Lettuce - Romaine, Heart
print("Quiz 11: PASS")


# Quiz 12 (sql_hr): Stored procedure promote_to_manager
def sp_promote_to_manager(emp_id: int, new_salary: float, new_reports_to: int | None) -> None:
    conn.execute(
        "UPDATE employees SET salary = ?, reports_to = ? WHERE employee_id = ?;",
        (new_salary, new_reports_to, emp_id),
    )


conn.execute("SAVEPOINT q12_test;")
sp_promote_to_manager(63196, 95000, 37270)
promoted = query("SELECT salary, reports_to FROM employees WHERE employee_id = 63196;")[0]
assert promoted["salary"] == 95000 and promoted["reports_to"] == 37270
conn.execute("ROLLBACK TO SAVEPOINT q12_test;")
conn.execute("RELEASE SAVEPOINT q12_test;")
print("Quiz 12: PASS")


# Quiz 13 (sql_invoicing): Stored procedure apply_discount
def sp_apply_discount(min_invoice_total: float, discount_rate: float) -> None:
    conn.execute(
        "UPDATE invoices SET invoice_total = ROUND(invoice_total * (1 - ?), 2) WHERE invoice_total > ?;",
        (discount_rate, min_invoice_total),
    )


conn.execute("SAVEPOINT q13_test;")
sp_apply_discount(200.0, 0.10)
assert (
    query("SELECT invoice_total FROM invoices WHERE invoice_id = 10;")[0]["invoice_total"] == 189.0
)
conn.execute("ROLLBACK TO SAVEPOINT q13_test;")
conn.execute("RELEASE SAVEPOINT q13_test;")
print("Quiz 13: PASS")

# Quiz 14 (sql_store): Order with highest total amount using a CTE
q14 = query("""
WITH order_totals AS (
    SELECT order_id, ROUND(SUM(quantity * unit_price), 2) AS total_amount
    FROM order_items
    GROUP BY order_id
)
SELECT order_id, total_amount
FROM order_totals
ORDER BY total_amount DESC
LIMIT 1;
""")
assert q14[0]["order_id"] == 10 and q14[0]["total_amount"] == 124.51
print("Quiz 14: PASS")

# Quiz 15 (sql_store): Temp table product_sales
conn.executescript("""
DROP TABLE IF EXISTS product_sales;
CREATE TEMPORARY TABLE product_sales AS
SELECT
    product_id,
    SUM(quantity) AS total_quantity_sold,
    ROUND(SUM(quantity * unit_price), 2) AS total_revenue
FROM order_items
GROUP BY product_id;
""")
assert len(query("SELECT * FROM product_sales;")) == 11
print("Quiz 15: PASS")


# Quiz 16 (sql_hr): Stored procedure calculate_office_salaries
def sp_calculate_office_salaries() -> list[dict]:
    conn.executescript("""
    DROP TABLE IF EXISTS office_salaries;
    CREATE TEMPORARY TABLE office_salaries (office_id INTEGER, total_salary REAL);
    """)
    for (office_id,) in conn.execute("SELECT office_id FROM offices ORDER BY office_id;"):
        total_sal = conn.execute(
            "SELECT COALESCE(SUM(salary), 0) FROM employees WHERE office_id = ?;",
            (office_id,),
        ).fetchone()[0]
        conn.execute("INSERT INTO office_salaries VALUES (?, ?);", (office_id, total_sal))
    return query("SELECT * FROM office_salaries ORDER BY office_id;")


q16 = sp_calculate_office_salaries()
assert len(q16) == 4 and q16[0]["total_salary"] == 538000.0
print("Quiz 16: PASS")

# Quiz 17 (sql_hr): Temp table manager_salaries
conn.executescript("""
DROP TABLE IF EXISTS manager_salaries;
CREATE TEMPORARY TABLE manager_salaries AS
SELECT
    manager.employee_id AS manager_id,
    manager.first_name AS manager_first_name,
    manager.last_name AS manager_last_name,
    SUM(employee.salary) AS total_salary
FROM employees AS manager
JOIN employees AS employee
    ON manager.employee_id = employee.reports_to
GROUP BY manager.employee_id, manager.first_name, manager.last_name;
""")
q17 = query("SELECT * FROM manager_salaries ORDER BY manager_id;")
assert len(q17) == 4
print("Quiz 17: PASS")

# Quiz 18 (sql_store): Products never ordered using subquery
q18 = query("""
SELECT product_id, name
FROM products
WHERE product_id NOT IN (SELECT product_id FROM order_items)
ORDER BY product_id;
""")
assert [r["product_id"] for r in q18] == [7, 12]
print("Quiz 18: PASS")

# Quiz 19 (sql_store): Customers who never placed an order using subquery
q19 = query("""
SELECT customer_id, first_name, last_name
FROM customers
WHERE customer_id NOT IN (SELECT customer_id FROM orders)
ORDER BY customer_id;
""")
assert [r["customer_id"] for r in q19] == [1, 3, 4, 9, 11]
print("Quiz 19: PASS")

# Quiz 20 (sql_hr): Employees earning more than their office average
q20 = query("""
SELECT e.employee_id, e.first_name, e.last_name, e.salary
FROM employees e
WHERE e.salary > (
    SELECT AVG(e2.salary)
    FROM employees e2
    WHERE e2.office_id = e.office_id
)
ORDER BY e.employee_id;
""")
assert [r["employee_id"] for r in q20] == [37270, 37851, 40448, 56274]
print("Quiz 20: PASS")

# Quiz 21 (sql_store): Customers who ordered every product at least once
q21 = query("""
SELECT c.customer_id, c.first_name, c.last_name
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.first_name, c.last_name
HAVING COUNT(DISTINCT oi.product_id) = (SELECT COUNT(*) FROM products);
""")
assert len(q21) == 0
print("Quiz 21: PASS")


# Quiz 22 (sql_hr): Stored procedure RaiseSalary
def sp_raise_salary(employee_id_param: int, percentage_increase: float) -> None:
    conn.execute(
        "UPDATE employees SET salary = ROUND(salary + salary * (? / 100.0), 2) WHERE employee_id = ?;",
        (percentage_increase, employee_id_param),
    )


conn.execute("SAVEPOINT q22_test;")
sp_raise_salary(67009, 10.0)
assert query("SELECT salary FROM employees WHERE employee_id = 67009;")[0]["salary"] == 49500
conn.execute("ROLLBACK TO SAVEPOINT q22_test;")
conn.execute("RELEASE SAVEPOINT q22_test;")
print("Quiz 22: PASS")

# Quiz 23 (sql_store): Customer who ordered the most distinct products
q23 = query("""
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    COUNT(DISTINCT oi.product_id) AS distinct_products_ordered
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.first_name, c.last_name
ORDER BY distinct_products_ordered DESC, c.customer_id ASC
LIMIT 1;
""")
assert q23[0]["customer_id"] == 5 and q23[0]["distinct_products_ordered"] == 5
print("Quiz 23: PASS")

# Quiz 24 (sql_hr): Offices with more than average number of employees
q24 = query("""
SELECT
    o.office_id,
    o.address,
    o.city,
    o.state,
    COUNT(e.employee_id) AS employee_count
FROM offices o
JOIN employees e ON o.office_id = e.office_id
GROUP BY o.office_id, o.address, o.city, o.state
HAVING employee_count > (
    SELECT AVG(employee_count)
    FROM (
        SELECT office_id, COUNT(employee_id) AS employee_count
        FROM employees
        GROUP BY office_id
    ) AS average_office
);
""")
assert len(q24) == 1 and q24[0]["office_id"] == 1 and q24[0]["employee_count"] == 7
print("Quiz 24: PASS")

# Quiz 25 (sql_store): Shipper that has shipped the most orders
q25 = query("""
SELECT
    s.shipper_id,
    s.name AS shipper_name,
    COUNT(o.order_id) AS order_count
FROM shippers s
JOIN orders o ON s.shipper_id = o.shipper_id
GROUP BY s.shipper_id, s.name
ORDER BY order_count DESC, s.shipper_id ASC
LIMIT 1;
""")
assert q25[0]["shipper_id"] in {2, 4} and q25[0]["order_count"] == 2
print("Quiz 25: PASS")
print("\nALL 25 SQL-3 QUIZZES PASSED!")